In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_oci.chat_models import ChatOCIGenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
import os

In [ ]:
service_endpoint = os.getenv("OCI_SERVICE_ENDPOINT")
compartment_id = os.getenv("OCI_COMPARTMENT_ID")
model_id = os.getenv("OCI_MODEL_ID")

llm = ChatOCIGenAI(
    service_endpoint=service_endpoint,
    compartment_id=compartment_id,
    model_id=model_id,
)

In [ ]:

class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [ ]:
def generate_joke(state: JokeState):
    prompt=f'generate a joke on the topic {state["topic"]}'
    response=llm.invoke(prompt).content
    return {'joke':response}

In [ ]:
def generate_explanation(state: JokeState):
    prompt=f'write an explanation for the joke - {state["joke"]}'
    response=llm.invoke(prompt).content
    return {'explanation':response}

In [ ]:

graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)
workflow

In [ ]:
config={"configurable":{"thread_id":"user_1"}}
workflow.invoke({'topic':'Indian hygine'},config)

In [ ]:
workflow.get_state(config)

In [ ]:
list(workflow.get_state_history(config))

In [ ]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

In [ ]:
list(workflow.get_state_history(config2))

Updating State

In [ ]:

workflow.update_state({"configurable":{"thread_id":"user_1","checkpoint_id":"1f0e3ebe-59f3-666a-8006-8aebe926faa5",'checkpoint_ns':""}},{"topic":"Democracy"})

In [ ]:
list(workflow.get_state_history(config))

In [ ]:
workflow.invoke(None,{"configurable":{"thread_id":"user_1","checkpoint_id":"1f0e3edf-820e-6fa5-8007-1d81fbfb48ab"}})

Fault Tolerance

In [ ]:
from langgraph.graph import StateGraph,END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:

# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))